# Pearl's Causal Ladder: An Interactive Introduction

This notebook provides a hands-on introduction to **Judea Pearl's causal hierarchy**
(the "Ladder of Causation") using synthetic data and visualizations.

We demonstrate why each rung requires progressively stronger assumptions and
different analytical machinery:

| Rung | Name | Question | Formal | Requires |
|------|------|----------|--------|----------|
| **L1** | Association | "What is?" | $P(Y \mid X)$ | Observational data |
| **L2** | Intervention | "What if I do?" | $P(Y \mid do(X))$ | Causal DAG + identification |
| **L3** | Counterfactual | "What if it had been different?" | $P(Y_x \mid X=x', Y=y')$ | Full SCM |

We also illustrate key pitfalls: confounding, collider bias, and the gap between
correlation and causation.

In [ ]:
import sys
from pathlib import Path

# Add the src directory to the path so we can import the causal_inference package
src_dir = Path.cwd().parent
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx

from causal_inference.data.synthetic import (
    generate_confounded_data,
    generate_frontdoor_data,
    generate_collider_data,
    generate_rct_data,
)
from causal_inference.viz.dag import draw_dag_from_spec

plt.style.use("seaborn-v0_8-whitegrid")
print("Setup complete.")

---

## 1. The Confounding Problem: Why Correlation ≠ Causation

We generate data from a **confounded** scenario:

```
Genetics ──→ Smoking ──→ Cancer
    └──────────────────→ Cancer
```

Genetics affects both Smoking and Cancer. The naive correlation between
Smoking and Cancer **overstates** the true causal effect because part of
the observed association is due to genetic predisposition.

In [ ]:
ds_conf = generate_confounded_data(n=3000, treatment_effect=0.3)

fig = draw_dag_from_spec(ds_conf.dag, figsize=(7, 4),
                         title="Confounded DAG: Genetics → Smoking → Cancer")
plt.show()

print(f"True causal effect (ATE) of Smoking on Cancer: {ds_conf.true_ate}")
print(f"Scenario: {ds_conf.description}")

In [ ]:
df = ds_conf.data

# Naive correlation (confounded)
naive_corr = df["smoking"].corr(df["cancer_score"])

# Partial correlation controlling for genetics (deconfounded)
from numpy.linalg import lstsq

def partial_corr(df, x, y, z):
    """Partial correlation of x and y given z."""
    # Residualize x on z
    Z = df[[z]].values
    ones = np.ones((len(df), 1))
    Z_aug = np.hstack([Z, ones])
    beta_x = lstsq(Z_aug, df[x].values, rcond=None)[0]
    resid_x = df[x].values - Z_aug @ beta_x
    # Residualize y on z
    beta_y = lstsq(Z_aug, df[y].values, rcond=None)[0]
    resid_y = df[y].values - Z_aug @ beta_y
    return np.corrcoef(resid_x, resid_y)[0, 1]

adjusted_corr = partial_corr(df, "smoking", "cancer_score", "genetics")

print(f"Naive correlation (Smoking, Cancer):              {naive_corr:.4f}")
print(f"Partial correlation (controlling for Genetics):   {adjusted_corr:.4f}")
print(f"True causal effect (ATE):                         {ds_conf.true_ate:.4f}")
print()
print("→ The naive correlation is inflated by confounding.")
print("  Adjusting for Genetics brings us closer to the true causal effect.")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: scatter showing raw (confounded) relationship
axes[0].scatter(df["smoking"], df["cancer_score"], alpha=0.2, s=10, color="#e17055")
z = np.polyfit(df["smoking"], df["cancer_score"], 1)
x_line = np.linspace(df["smoking"].min(), df["smoking"].max(), 100)
axes[0].plot(x_line, np.polyval(z, x_line), "r-", linewidth=2,
             label=f"Naive slope = {z[0]:.3f}")
axes[0].set_xlabel("Smoking")
axes[0].set_ylabel("Cancer Score")
axes[0].set_title("Naive (Confounded) Relationship")
axes[0].legend()

# Right: comparison bar chart
labels = ["Naive\nCorrelation", "Adjusted\n(Partial Corr)", "True\nATE"]
values = [naive_corr, adjusted_corr, ds_conf.true_ate]
colors = ["#e17055", "#00b894", "#0984e3"]
bars = axes[1].bar(labels, values, color=colors, edgecolor="#2d3436", linewidth=1.2)
axes[1].set_ylabel("Effect Size")
axes[1].set_title("Confounding Inflates the Naive Estimate")
for bar, val in zip(bars, values):
    axes[1].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
                 f"{val:.3f}", ha="center", fontweight="bold")

fig.suptitle("Rung 1 vs Rung 2: Correlation ≠ Causation", fontsize=14, fontweight="bold", y=1.02)
fig.tight_layout()
plt.show()

---

## 2. Rung 1 — Association: $P(Y \mid X)$

At **Rung 1** we can only ask: *"What is the probability of Y given that we observe X?"*

This is the domain of **statistics**: regression, correlation, prediction.
No causal claims are possible at this level.

### What Rung 1 Can Do
- Predict outcomes from observed features
- Identify statistical associations
- Build machine learning models

### What Rung 1 Cannot Do
- Distinguish genuine causes from spurious correlations
- Predict the effect of interventions
- Answer "what if" questions

In [ ]:
from scipy import stats

# L1 analysis: conditional distribution P(Cancer | Smoking)
slope, intercept, r_value, p_value, std_err = stats.linregress(
    df["smoking"], df["cancer_score"]
)

print("═" * 50)
print("RUNG 1: ASSOCIATION ANALYSIS")
print("═" * 50)
print(f"\nRegression: Cancer = {intercept:.3f} + {slope:.3f} × Smoking")
print(f"R² = {r_value**2:.4f}")
print(f"p-value = {p_value:.2e}")
print(f"\n⚠️  This slope ({slope:.3f}) ≠ true causal effect ({ds_conf.true_ate})")
print("   because Rung 1 cannot separate causation from confounding.")

---

## 3. Rung 2 — Intervention: $P(Y \mid do(X))$

At **Rung 2** we ask: *"What would happen if we intervened to set X to a particular value?"*

This requires a **causal DAG** and **identification** of the causal effect.
The key insight: $P(Y \mid X) \neq P(Y \mid do(X))$ in general.

The `do`-operator severs all incoming edges to the treatment variable,
breaking the confounding pathways.

### Identification Strategies
- **Back-door criterion**: adjust for a sufficient set of confounders
- **Front-door criterion**: use a mediator that blocks confounding
- **Instrumental variables**: use an exogenous source of variation

In [ ]:
# L2 analysis: adjusting for the confounder (back-door criterion)
from numpy.linalg import lstsq

X_adj = np.column_stack([df["smoking"].values, df["genetics"].values, np.ones(len(df))])
betas = lstsq(X_adj, df["cancer_score"].values, rcond=None)[0]

print("═" * 50)
print("RUNG 2: INTERVENTIONAL ANALYSIS")
print("═" * 50)
print(f"\nBack-door adjustment (controlling for Genetics):")
print(f"  Cancer = {betas[2]:.3f} + {betas[0]:.3f} × Smoking + {betas[1]:.3f} × Genetics")
print(f"\n  Estimated causal effect of Smoking: {betas[0]:.4f}")
print(f"  True causal effect (ATE):            {ds_conf.true_ate:.4f}")
print(f"\n✓ Back-door adjustment recovers the true causal effect!")

### Front-Door Criterion

When the confounder is **unmeasured**, we can sometimes use the front-door
criterion if there is a mediator that is:
1. Fully mediated by the treatment (no direct effect of confounder on mediator)
2. The only path from treatment to outcome

```
Genetics (unmeasured)
  ├──→ Smoking ──→ Tar ──→ Cancer
  └─────────────────────→ Cancer
```

In [ ]:
ds_fd = generate_frontdoor_data(n=3000)

fig = draw_dag_from_spec(ds_fd.dag, figsize=(8, 4),
                         title="Front-Door: Smoking → Tar → Cancer (Genetics unmeasured)")
plt.show()

df_fd = ds_fd.data

# Step 1: Estimate Smoking → Tar (no confounding for this path)
X1 = np.column_stack([df_fd["smoking"].values, np.ones(len(df_fd))])
b_smoke_tar = lstsq(X1, df_fd["tar"].values, rcond=None)[0]

# Step 2: Estimate Tar → Cancer (adjusting for Smoking to block backdoor)
X2 = np.column_stack([df_fd["tar"].values, df_fd["smoking"].values, np.ones(len(df_fd))])
b_tar_cancer = lstsq(X2, df_fd["cancer_score"].values, rcond=None)[0]

# Front-door estimate = product of the two path coefficients
fd_estimate = b_smoke_tar[0] * b_tar_cancer[0]

print("═" * 50)
print("FRONT-DOOR CRITERION")
print("═" * 50)
print(f"\nStep 1: Smoking → Tar coefficient: {b_smoke_tar[0]:.4f}")
print(f"Step 2: Tar → Cancer coefficient:  {b_tar_cancer[0]:.4f}")
print(f"\nFront-door estimate (product):     {fd_estimate:.4f}")
print(f"True causal effect (ATE):           {ds_fd.true_ate:.4f}")
print(f"\n✓ The front-door criterion identifies the causal effect")
print("  even though Genetics is unmeasured!")

---

## 4. Collider Bias: A Pitfall from Conditioning on the Wrong Variable

```
Talent ──→ Success ←── Beauty
```

Talent and Beauty are **independent** in the population. But among successful
people (i.e., conditioning on the collider `Success`), they appear
**negatively correlated** — this is **Berkson's paradox**.

In [ ]:
ds_coll = generate_collider_data(n=5000)

fig = draw_dag_from_spec(ds_coll.dag, figsize=(6, 4),
                         title="Collider: Talent → Success ← Beauty")
plt.show()

df_c = ds_coll.data

# Marginal correlation (correct: near zero)
marginal_corr = df_c["talent"].corr(df_c["beauty"])

# Conditional correlation among successful people (collider bias)
successful = df_c[df_c["is_successful"] == 1]
conditional_corr = successful["talent"].corr(successful["beauty"])

print("═" * 50)
print("COLLIDER BIAS (Berkson's Paradox)")
print("═" * 50)
print(f"\nCorr(Talent, Beauty) — overall:          {marginal_corr:.4f}  (≈ 0, independent)")
print(f"Corr(Talent, Beauty) — among successful:  {conditional_corr:.4f}  (negative!)")
print(f"\n⚠️  Conditioning on a collider creates spurious associations.")
print("   'Among successful people, the beautiful tend to be less talented'")
print("   is a selection effect, not a causal relationship.")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Full population
axes[0].scatter(df_c["talent"], df_c["beauty"], alpha=0.15, s=8, color="#636e72")
axes[0].set_xlabel("Talent")
axes[0].set_ylabel("Beauty")
axes[0].set_title(f"Full Population (r = {marginal_corr:.3f})")

# Conditioned on success
axes[1].scatter(successful["talent"], successful["beauty"], alpha=0.2, s=8, color="#e17055")
z_coll = np.polyfit(successful["talent"], successful["beauty"], 1)
x_line = np.linspace(successful["talent"].min(), successful["talent"].max(), 100)
axes[1].plot(x_line, np.polyval(z_coll, x_line), "r-", linewidth=2)
axes[1].set_xlabel("Talent")
axes[1].set_ylabel("Beauty")
axes[1].set_title(f"Among Successful Only (r = {conditional_corr:.3f})")

fig.suptitle("Collider Bias: Conditioning on Success Creates Spurious Correlation",
             fontsize=13, fontweight="bold", y=1.02)
fig.tight_layout()
plt.show()

---

## 5. Rung 3 — Counterfactual: $P(Y_x \mid X=x', Y=y')$

At **Rung 3** we ask: *"Given that we observed X=x' and Y=y', what would Y
have been if X had been x instead?"*

This requires a **fully specified Structural Causal Model (SCM)**:
- Structural equations: $V_i = f_i(\text{Pa}(V_i), U_i)$
- Exogenous distributions: $P(U)$

### The Three-Step Procedure (Pearl)

1. **Abduction**: given observed data, infer the exogenous noise $U$
2. **Action**: modify the SCM by applying $do(X = x)$
3. **Prediction**: propagate forward through the modified model

### Example: Drug Treatment Counterfactual

SCM:
- $X = U_X$ where $U_X \sim \text{Bernoulli}(0.5)$ (treatment assignment)
- $Y = 0.6 \cdot X + U_Y$ where $U_Y \sim N(0, 0.3)$ (recovery score)

**Observed**: Patient received treatment ($X = 1$) and recovered with $Y = 0.7$.

**Counterfactual question**: What would $Y$ have been had the patient **not** received treatment ($X = 0$)?

In [ ]:
# Counterfactual computation using Pearl's three-step procedure

observed_X = 1
observed_Y = 0.7
beta = 0.6

print("═" * 50)
print("RUNG 3: COUNTERFACTUAL ANALYSIS")
print("═" * 50)

# Step 1: ABDUCTION
# From Y = 0.6 * X + U_Y, we get U_Y = Y - 0.6 * X
U_Y_inferred = observed_Y - beta * observed_X
print(f"\nStep 1 — Abduction:")
print(f"  Y = {beta} × X + U_Y")
print(f"  {observed_Y} = {beta} × {observed_X} + U_Y")
print(f"  U_Y = {U_Y_inferred}")

# Step 2: ACTION
counterfactual_X = 0
print(f"\nStep 2 — Action:")
print(f"  Set X = {counterfactual_X} (do-operator)")

# Step 3: PREDICTION
counterfactual_Y = beta * counterfactual_X + U_Y_inferred
print(f"\nStep 3 — Prediction:")
print(f"  Y_{{X=0}} = {beta} × {counterfactual_X} + {U_Y_inferred} = {counterfactual_Y}")

print(f"\n{'─' * 50}")
print(f"Conclusion: Had the patient NOT received treatment,")
print(f"their recovery score would have been {counterfactual_Y:.2f}")
print(f"instead of the observed {observed_Y:.2f}.")
print(f"\nThe individual treatment effect for this patient: {observed_Y - counterfactual_Y:.2f}")

In [ ]:
# Visualize: distribution of counterfactual outcomes across a population
ds_rct = generate_rct_data(n=2000, true_effect=0.6)
df_rct = ds_rct.data

treated = df_rct[df_rct["treatment"] == 1]
control = df_rct[df_rct["treatment"] == 0]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Distribution of outcomes
axes[0].hist(treated["recovery"], bins=30, alpha=0.6, color="#0984e3", label="Treated (X=1)", density=True)
axes[0].hist(control["recovery"], bins=30, alpha=0.6, color="#e17055", label="Control (X=0)", density=True)
axes[0].axvline(treated["recovery"].mean(), color="#0984e3", linestyle="--", linewidth=2)
axes[0].axvline(control["recovery"].mean(), color="#e17055", linestyle="--", linewidth=2)
axes[0].set_xlabel("Recovery Score")
axes[0].set_ylabel("Density")
axes[0].set_title("Observed Outcome Distributions")
axes[0].legend()

# Individual counterfactuals for treated patients
# For each treated patient: U_Y = Y - 0.6 * 1, counterfactual Y = 0 + U_Y
treated_cf = treated["recovery"].values - 0.6  # counterfactual: what if X=0
ite = treated["recovery"].values - treated_cf   # individual treatment effects

axes[1].hist(ite, bins=30, alpha=0.7, color="#00b894", edgecolor="#2d3436")
axes[1].axvline(ite.mean(), color="#e17055", linestyle="--", linewidth=2,
                label=f"Mean ITE = {ite.mean():.3f}")
axes[1].axvline(0.6, color="#0984e3", linestyle=":", linewidth=2,
                label=f"True ATE = 0.600")
axes[1].set_xlabel("Individual Treatment Effect (ITE)")
axes[1].set_ylabel("Count")
axes[1].set_title("Rung 3: Counterfactual Individual Effects")
axes[1].legend()

fig.suptitle("From Population Averages (Rung 2) to Individual Counterfactuals (Rung 3)",
             fontsize=13, fontweight="bold", y=1.02)
fig.tight_layout()
plt.show()

---

## 6. Summary: The Causal Ladder

| Rung | Question | Machinery | This Notebook |
|------|----------|-----------|---------------|
| **L1** | Is Smoking correlated with Cancer? | Regression, correlation | Naive slope overstates effect |
| **L2** | Does Smoking *cause* Cancer? | DAG + back-door / front-door | Adjusting for Genetics recovers true ATE |
| **L3** | Had this patient not smoked, would they have gotten cancer? | Full SCM + abduction | Three-step procedure gives individual answer |

### Key Takeaways

1. **Correlation ≠ causation** — confounders inflate naive estimates (Section 1)
2. **Identification requires a DAG** — the back-door and front-door criteria are structural, not statistical (Section 3)
3. **Conditioning on colliders creates bias** — not all adjustments improve estimates (Section 4)
4. **Counterfactuals require a full SCM** — DAGs alone are insufficient for Rung 3 (Section 5)
5. **The agentic workflow** in the companion notebooks automates the classification of questions and the selection of appropriate rung-level analysis